# Chapter 7 - Lab 3: A conversational analyst with AutoGen (customer-care edition)

**Goal**: a two-agent **conversation** in which an LLM assistant writes Python (pandas, matplotlib) to analyse a
ticket export, a code executor runs it, and the assistant reads the output and iterates.

*Finance edition*: fetch NVIDIA prices with yfinance, compute metrics, plot 6 months of prices.
*Customer-care edition*: analyse a quarter of **support tickets**: weekly volume by channel, CSAT and
resolution time by category, the categories that hurt CSAT most, and a saved plot.

**Tech stack**: the classic AutoGen 0.2-style API (`AssistantAgent` + `UserProxyAgent` with a code executor),
OpenAI `gpt-4o`, pandas and matplotlib.

The finance edition installed `pyautogen<0.4`. On current Python versions that pin now resolves to a very old
release, so we install **AG2** (`ag2<1.0`), the community continuation of the AutoGen 0.2 line: same imports
(`from autogen import AssistantAgent, UserProxyAgent`), same behaviour. (AG2 1.x introduced a new API.)

## 1. Install packages

In [ ]:
%pip install -q "ag2[openai]<1.0" pandas matplotlib python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Create the ticket export

In [ ]:
import os, sys, tempfile
import numpy as np
import pandas as pd

# Make the code executor's `python` this kernel's interpreter (same packages as the notebook).
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

work_dir = tempfile.mkdtemp()

# Synthetic Q3 ticket export (13 weeks, 4 channels, 6 categories). Replace with your ticketing export.
rng = np.random.default_rng(7)
n = 2400
weeks = pd.date_range("2026-07-06", periods=13, freq="W-MON")
categories = ["Where is my order", "Product defect", "Installation", "Billing", "Returns", "How-to"]
cat_p = [0.30, 0.22, 0.14, 0.12, 0.12, 0.10]
channels = ["Phone", "WhatsApp", "Email", "Chat"]
df = pd.DataFrame({
    "ticket_id": [f"TKT-{60000 + i}" for i in range(n)],
    "week": rng.choice(weeks, size=n, p=np.linspace(0.6, 1.4, 13) / np.linspace(0.6, 1.4, 13).sum()),
    "channel": rng.choice(channels, size=n, p=[0.35, 0.30, 0.20, 0.15]),
    "category": rng.choice(categories, size=n, p=cat_p),
})
base_csat = {"Where is my order": 3.6, "Product defect": 3.1, "Installation": 3.9,
             "Billing": 4.0, "Returns": 3.7, "How-to": 4.6}
base_hours = {"Where is my order": 20, "Product defect": 70, "Installation": 30,
              "Billing": 12, "Returns": 40, "How-to": 3}
df["csat"] = (df["category"].map(base_csat) + rng.normal(0, 0.6, n)).clip(1, 5).round(1)
df["resolution_hours"] = (df["category"].map(base_hours) * rng.lognormal(0, 0.4, n)).round(1)
df.to_csv(os.path.join(work_dir, "tickets_q3.csv"), index=False)
print(f"Wrote {len(df)} tickets to {work_dir}/tickets_q3.csv")
df.head()

## 4. Configure the model and the code executor

`cache_seed` enables AutoGen's response cache (cheap, reproducible re-runs). The executor runs generated code inside `work_dir`, where the CSV lives and plots are saved.

In [ ]:
from autogen.coding import LocalCommandLineCodeExecutor

config_list = [{"model": "gpt-4o", "api_key": os.environ["OPENAI_API_KEY"]}]
llm_config = {"config_list": config_list, "cache_seed": 42}

executor = LocalCommandLineCodeExecutor(timeout=60, work_dir=work_dir)
print(f"Code executor working directory: {work_dir}")

## 5. Define the two agents

Two rules in the system message keep the conversation well-behaved: plots are **saved to files** (no display), and `TERMINATE` may only be written **after** the assistant has seen execution results.

In [ ]:
from autogen import AssistantAgent, UserProxyAgent

assistant = AssistantAgent(
    name="assistant",
    llm_config=llm_config,
    system_message=(
        "You are a customer-experience data analyst. When asked to analyse tickets, write Python code "
        "with pandas to read the CSV in the current directory and compute the requested metrics. Present "
        "results in a clear, structured format with business interpretation. Save any plot to a PNG file "
        "instead of calling plt.show(). After you have seen the execution results and written your final "
        "analysis, end that final message with TERMINATE. Never write TERMINATE in a message that contains code."
    ),
)

code_executor = UserProxyAgent(
    name="code_executor",
    human_input_mode="NEVER",
    code_execution_config={"executor": executor},
    llm_config=False,
    is_termination_msg=lambda msg: "TERMINATE" in (msg.get("content") or ""),
)

## 6. Run the conversation

In [ ]:
query = ("Analyse tickets_q3.csv: (1) weekly ticket volume by channel and the overall trend; (2) average CSAT "
         "and average resolution hours by category; (3) the three categories that hurt CSAT most, weighted "
         "by volume. Plot weekly ticket volume by channel and save it to weekly_volume.png.")

chat_result = code_executor.initiate_chat(assistant, message=query, max_turns=5)

## 7. Final answer, cost and plots

In [ ]:
print("FINAL ANALYSIS:")
print(chat_result.summary.replace("TERMINATE", "").strip())
print("\nCOST:")
print(chat_result.cost)

In [ ]:
import glob
from IPython.display import Image, display

for png in sorted(glob.glob(os.path.join(work_dir, "*.png"))):
    print(png)
    display(Image(filename=png))

## 8. Results

**What to notice about conversational multi-agent systems**
* **Conversation is the coordination mechanism**: no handoffs, no shared state object.
* **Code execution closes the loop**: the assistant sees real output (or a traceback) and fixes its own code.
* **Termination needs care**: `TERMINATE` rules plus a hard `max_turns`.
* **Flexible but less predictable**: ideal for exploratory analysis by a CX analyst; harder to audit
  than a fixed pipeline. Never give a code-executing agent access to production systems.